# Galerie de réentraînement — Cas santé Médicia+

> **Contexte** : tu rejoues le pattern d'entraînement vu en M1-B1, sur un nouveau client, un nouveau domaine métier, un nouveau dataset.
> Objectif : ancrer C5 par la répétition.

---

## 🎬 Situation

**Vendredi 17h chez FastIA.** Karim te transfère un mail :

> **De :** Dr. Léa Mercier, Direction Médicale — Médicia+ (mutuelle santé)
> **Objet :** Modèle d'aide au tri des biopsies du sein
>
> Bonjour FastIA,
>
> Nous souhaitons explorer un modèle d'**aide au tri** des biopsies du sein
> à partir de mesures morphologiques cellulaires. L'idée n'est pas de
> remplacer un anatomopathologiste, mais d'**ordonnancer les lames** : les
> cas les plus suspects en haut de la pile pour réduire les délais de
> rendu sur les vraies urgences.
>
> Nous disposons d'un jeu de données public anonymisé (569 biopsies,
> 30 indicateurs morphologiques mesurés au microscope). Pouvez-vous :
> 1. Entraîner un modèle de classification binaire (malin / bénin) ;
> 2. Comparer 2 configurations d'hyperparamètres ;
> 3. Nous rendre un verdict argumenté chiffré.
>
> Cordialement,
> Dr. Léa Mercier.

---

## 🧭 Pattern mobilisé

Le même que M1-B1, en 6 étapes :

1. **Charger** le dataset
2. **Explorer** (EDA mini)
3. **Découper** train/test stratifié
4. **Entraîner** — 2 configurations d'hyperparamètres
5. **Évaluer** sur le test
6. **Persister** le modèle

Garde la fiche A4 `fiche_pattern_ML_supervise.md` ouverte à côté.

## Setup

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    f1_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## [1] Charger le dataset

Source : **Breast Cancer Wisconsin (Diagnostic)** — embarqué directement
dans scikit-learn, donc **pas de téléchargement ni d'internet requis**.

569 biopsies, 30 indicateurs morphologiques mesurés au microscope sur
des noyaux cellulaires (rayon, texture, périmètre, surface, lissé, etc.)
— déclinés en moyenne, écart-type et valeur la plus extrême par image.
Tu n'as **pas besoin** de comprendre la signification clinique de chaque
indicateur pour entraîner le modèle.

**Cible** : `1` = malin (à orienter en priorité) / `0` = bénin.

> ⚠️ La cible par défaut de scikit-learn est inversée par rapport à la
> convention clinique (sklearn met `1` pour bénin). On la retourne pour
> que `1` corresponde au cas positif à détecter — c'est plus lisible côté
> matrice de confusion et plus cohérent avec ce que tu as vu en M1-B1
> (où `1` = défaut crédit).

In [ ]:
data = load_breast_cancer(as_frame=True)
X = data.data
y = pd.Series(1 - data.target, name="target")

df = X.assign(target=y)

print(f"Lignes : {df.shape[0]} | Colonnes : {df.shape[1]}")
df.iloc[:, list(range(4)) + [-1]].head()

## [2] Explorer (EDA mini)

Trois choses à vérifier avant tout :
1. **Valeurs manquantes** ? Si oui, combien.
2. **Équilibre des classes** ? Crucial pour choisir la métrique.
3. **Distribution de quelques variables clés** ? Pour repérer des aberrations.

On reste **mini** — 15 min max. Le métier (interprétation médicale) n'est pas
ton sujet ici.

In [ ]:
print(f"Valeurs manquantes au total : {df.isna().sum().sum()}")

print("\nÉquilibre de la cible :")
print(y.value_counts().rename({0: "bénin", 1: "malin"}))
print(f"Ratio de la classe positive (malin) : {y.mean():.2%}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.countplot(x=y.map({0: "bénin", 1: "malin"}), ax=axes[0])
axes[0].set_title("Équilibre bénin vs malin")
axes[0].set_xlabel("")

sns.histplot(
    data=df,
    x="mean radius",
    hue=df["target"].map({0: "bénin", 1: "malin"}),
    kde=True,
    ax=axes[1],
)
axes[1].set_title("Rayon moyen des noyaux par classe")

sns.boxplot(
    x=df["target"].map({0: "bénin", 1: "malin"}),
    y=df["mean concavity"],
    ax=axes[2],
)
axes[2].set_title("Concavité moyenne par classe")
axes[2].set_xlabel("")

plt.tight_layout()
plt.show()

**Lecture rapide** : les classes sont à peu près équilibrées (~37%
positifs), donc F1 macro et accuracy resteront cohérents. Les biopsies
malignes ont des noyaux plus gros et plus concaves — c'est attendu
cliniquement et c'est un signal exploitable par le modèle.

## [3] Découper train / test

Split stratifié et reproductible — comme en M1-B1.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print(f"Train : {X_train.shape[0]} lignes | Test : {X_test.shape[0]} lignes")
print(f"Ratio positifs train : {y_train.mean():.2%}")
print(f"Ratio positifs test  : {y_test.mean():.2%}")

## [4] Entraîner — un plancher + 2 configurations

Avant de comparer des RandomForest entre eux, on se donne un **plancher** :
un `DummyClassifier` qui prédit toujours la classe majoritaire (bénin). Si nos
vrais modèles ne le battent pas franchement, c'est qu'ils n'apprennent rien.

Puis deux RandomForest :

- **Config A** : valeurs par défaut scikit-learn — référence.
- **Config B** : `n_estimators=300`, `max_depth=8`, `min_samples_leaf=3` —
  un peu plus de capacité, mais bridée pour limiter l'overfitting sur un
  petit dataset (569 lignes).

> ⚠️ **Pas de `StandardScaler` ici.** Un RandomForest (comme tout modèle à
> base d'arbres) découpe les features par seuils : il est **insensible à leur
> échelle**. Mettre un scaler devant ne change rien — on le réserve aux
> modèles **linéaires** (tu en verras un, le `Ridge`, dans le notebook 03).

In [ ]:
# Plancher naïf : prédit toujours la classe majoritaire (bénin).
dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)

# Pas de scaler : un RandomForest est insensible à l'échelle des features.
clf_a = RandomForestClassifier(random_state=RANDOM_STATE)
clf_b = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=3,
)

clf_a.fit(X_train, y_train)
clf_b.fit(X_train, y_train)
print("Plancher + deux modèles entraînés.")

## [5] Évaluer sur le test

Métriques : F1 macro (sensible aux deux classes), ROC-AUC (capacité à
ordonner les biopsies par risque — clé pour le tri visé), matrice de
confusion.

In [ ]:
def evaluate(name: str, model: object) -> dict:
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    return {
        "modèle": name,
        "f1_macro": f1_score(y_test, y_pred, average="macro"),
        "roc_auc": roc_auc_score(y_test, y_proba),
    }


scores = pd.DataFrame(
    [
        evaluate("Plancher — Dummy", dummy),
        evaluate("Config A — défaut", clf_a),
        evaluate("Config B — tuned", clf_b),
    ]
)
scores.round(3)

In [ ]:
# On choisit le meilleur parmi les RandomForest (le plancher ne compte pas).
rf_scores = scores[scores["modèle"].str.startswith("Config")]
best_name = rf_scores.loc[rf_scores["f1_macro"].idxmax(), "modèle"]
best_model = clf_b if "B" in best_name else clf_a

print(f"Modèle retenu : {best_name}\n")
print(
    classification_report(
        y_test,
        best_model.predict(X_test),
        target_names=["bénin", "malin"],
    )
)

ConfusionMatrixDisplay.from_estimator(
    best_model, X_test, y_test, display_labels=["bénin", "malin"]
)
plt.title(f"Matrice de confusion — {best_name}")
plt.show()

## 🤔 Question réflexive — le coût des erreurs n'est pas symétrique

Dans un outil de **tri**, un **faux négatif** (biopsie maligne classée bénigne,
donc reléguée en bas de pile) n'a pas du tout le même coût qu'un **faux positif**
(biopsie bénigne priorisée : un anatomopathologiste relit une lame finalement
sans gravité).

Regarde ta matrice de confusion : combien de **FN** ? combien de **FP** ?
Lequel veux-tu minimiser en priorité ici — et qu'est-ce que ça impliquerait sur
le **seuil de décision** (0,5 par défaut) ? *(On ne le change pas dans ce
notebook, mais en santé c'est souvent là que se joue la vraie décision.)*

## [6] Persister le modèle

On sauve le modèle retenu avec `joblib`.

> 💡 Ici le modèle est un RandomForest seul (pas de préprocessing à embarquer).
> Dès qu'un modèle est précédé d'un scaler ou d'un encodeur (cf. le `Ridge` du
> notebook 03), tu sauves le **`Pipeline` entier** — sinon les transformations
> ne sont pas rejouées au moment de la prédiction.

In [ ]:
models_dir = Path("models")
models_dir.mkdir(exist_ok=True)

model_path = models_dir / "medicia_tri_biopsies_v1.joblib"
joblib.dump(best_model, model_path, compress=3)

print(f"Modèle sauvé : {model_path}")
print(f"Taille : {model_path.stat().st_size / 1024:.1f} Ko")

## 📝 Verdict — à remettre à Dr. Mercier

> Sur 114 biopsies de test (20 % du dataset), le modèle retenu (la meilleure
> des 2 configs, cf. tableau ci-dessus) atteint un F1 macro et un ROC-AUC
> élevés (> 0.95). Le nombre de **faux négatifs** (biopsies malignes classées
> bénignes) est très faible — ce qui est le point critique pour un outil de
> tri à visée d'aide au diagnostic.

> **Limites** : le jeu de données est issu d'un protocole de mesure
> spécifique (Université du Wisconsin). Avant tout usage sur les biopsies
> Médicia+, il faudrait vérifier que les indicateurs morphologiques sont
> mesurés de la même manière (sinon, écart de distribution garanti) et
> recalibrer le seuil de décision selon la tolérance clinique au faux
> négatif. Et bien sûr, **aucun usage diagnostique sans validation par
> un anatomopathologiste**.

---

## 🔎 Ce que tu viens de revoir

- Le pattern 6 étapes **transpose** d'un cas crédit (M1-B1, Pyrenex) à un
  cas santé (Médicia+) sans changer de logique.
- Le dataset est petit (569 vs 30 000 lignes en M1-B1) — la **taille du
  dataset** influence le choix d'hyperparamètres (`max_depth` plus contraint).
- Les métriques ne changent pas : F1 macro, ROC-AUC, matrice de confusion
  fonctionnent partout en classification binaire.
- La **convention de la cible** (1 = positif clinique vs 1 = bénin) est un
  point de vigilance — bien la fixer dès le début évite des matrices de
  confusion contre-intuitives.

## ⭐ Pour aller plus loin (optionnel)

- Remplace `RandomForestClassifier` par `GradientBoostingClassifier` —
  qu'est-ce qui change dans le code ? Dans les scores ?
- Ajoute une 3ème config qui force `class_weight="balanced"` et observe
  l'effet sur la matrice de confusion.
- Tente un `cross_val_score` à 5 folds sur la config retenue — la variance
  entre folds te dit si le score test est fiable malgré la petite taille.

---

*Galerie de réentraînement — cas santé Médicia+.*